
# RQ2 Formal Experiment 3 — Fed-CGM+Time+Image


In [13]:

from pathlib import Path
import sys, json, random
import numpy as np
import pandas as pd
import torch

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/dissertation")
else:
    PROJECT_ROOT = Path("../../..").resolve()

assert PROJECT_ROOT.exists(), f"PROJECT_ROOT does not exist: {PROJECT_ROOT}"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import dataloader as data
from src import federated as fed
from src import metrics
from src import train
from src.models.classical_baseline import HORIZONS
from src.models.multimodal import CGMTimePredictor
from src.models.multimodal_residual import FrozenImageOnlyResidualPredictor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]

TUNING_DIR = (
    PROJECT_ROOT / "experiments" / "7_federated_learning" / "results"
    / "configuration_tuning" / "fed_cgm_multiseed"
)
SELECTED_CONFIG_PATH = TUNING_DIR / "selected_fl_configuration.json"
assert SELECTED_CONFIG_PATH.exists(), f"Missing: {SELECTED_CONFIG_PATH}"

with open(SELECTED_CONFIG_PATH) as f:
    FL_CONFIG = json.load(f)

LOCAL_EPOCHS = int(FL_CONFIG["local_epochs"])
CLIENTS_PER_ROUND = int(FL_CONFIG["clients_per_round"])
EARLY_STOP_PATIENCE = int(FL_CONFIG["early_stop_patience"])
MAX_ROUNDS = int(FL_CONFIG["max_rounds"])
LR_PATIENCE = int(FL_CONFIG["lr_patience"])
LR_FACTOR = float(FL_CONFIG["lr_factor"])
# LR = float(FL_CONFIG["learning_rate"])
# WEIGHT_DECAY = float(FL_CONFIG["weight_decay"])
BATCH_SIZE = int(FL_CONFIG["batch_size"])
MAX_GRAD_NORM = float(FL_CONFIG["max_grad_norm"])

GRU_HIDDEN_SIZE = int(FL_CONFIG["gru_hidden_size"])
GRU_NUM_LAYERS = int(FL_CONFIG["gru_num_layers"])
GRU_DROPOUT = float(FL_CONFIG["gru_dropout"])

assert (LOCAL_EPOCHS, CLIENTS_PER_ROUND, EARLY_STOP_PATIENCE) == (5, 15, 20)

# RQ1 locked residual architectures.
# img_proj_dim/dropout are architecture choices from the final RQ1 experiment.
ENCODERS = {
    "resnet18": {
        "display_name": "ResNet18",
        "img_feat_dim": 512,
        "img_proj_dim": 4,
        "img_dropout": 0.0,
        "lr": 1e-3,
        "weight_decay": 1e-5,
    },
    "vit": {
        "display_name": "ViT-B/16",
        "img_feat_dim": 768,
        "img_proj_dim": 4,
        "img_dropout": 0.0,
        "lr": 1e-3,
        "weight_decay": 1e-5,
    },
    "clip": {
        "display_name": "CLIP ViT-B/16",
        "img_feat_dim": 768,
        "img_proj_dim": 4,
        "img_dropout": 0.0,
        "lr": 1e-3,
        "weight_decay": 1e-5,
    },
    "dinov2": {
        "display_name": "DINOv2 ViT-B/14",
        "img_feat_dim": 768,
        "img_proj_dim": 4,
        "img_dropout": 0.3,
        "lr": 1e-3,
        "weight_decay": 1e-5,
    },
}

EXPERIMENT_DIR = PROJECT_ROOT / "experiments" / "7_federated_learning"
BASELINE_CHECKPOINT_DIR = EXPERIMENT_DIR / "checkpoints" / "fed_cgm_time"
BASELINE_RESULTS_PATH = EXPERIMENT_DIR / "results" / "fed_cgm_time" / "run_results.json"

RESULTS_ROOT = EXPERIMENT_DIR / "results" / "fed_cgm_time_image"
CHECKPOINT_ROOT = EXPERIMENT_DIR / "checkpoints" / "fed_cgm_time_image"
SUMMARY_ROOT = RESULTS_ROOT / "summary"

for p in [RESULTS_ROOT, CHECKPOINT_ROOT, SUMMARY_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

assert BASELINE_CHECKPOINT_DIR.exists(), f"Missing Fed-CGM+Time checkpoints: {BASELINE_CHECKPOINT_DIR}"
assert BASELINE_RESULTS_PATH.exists(), f"Missing Fed-CGM+Time results: {BASELINE_RESULTS_PATH}"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Device:", DEVICE)
print(f"Locked FL protocol: E={LOCAL_EPOCHS}, C={CLIENTS_PER_ROUND}/30, P={EARLY_STOP_PATIENCE}")
# print("Locked FL optimizer: lr=", LR, "weight_decay=", WEIGHT_DECAY)
display(pd.DataFrame(ENCODERS).T)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_ROOT: /content/drive/MyDrive/dissertation
Device: cuda
Locked FL protocol: E=5, C=15/30, P=20


,display_name,img_feat_dim,img_proj_dim,img_dropout,lr,weight_decay
resnet18,ResNet18,512,4,0.0,0.001,0.00001
vit,ViT-B/16,768,4,0.0,0.001,0.00001
clip,CLIP ViT-B/16,768,4,0.0,0.001,0.00001
dinov2,DINOv2 ViT-B/14,768,4,0.3,0.001,0.00001


In [14]:

TIME_PROJ_DIM = 16
HEAD_DROPOUT = 0.2

def make_cgm_time_model():
    return CGMTimePredictor(
        gru_hidden_size=GRU_HIDDEN_SIZE,
        gru_num_layers=GRU_NUM_LAYERS,
        gru_dropout=GRU_DROPOUT,
        time_proj_dim=TIME_PROJ_DIM,
        head_dropout=HEAD_DROPOUT,
        num_horizons=len(HORIZONS),
    )

def find_best_checkpoint(seed):
    seed_dir = BASELINE_CHECKPOINT_DIR / f"seed{seed}"
    if not seed_dir.exists():
        # Defensive fallback in case an older run used seed_<n>.
        alt = BASELINE_CHECKPOINT_DIR / f"seed_{seed}"
        if alt.exists():
            seed_dir = alt
        else:
            raise FileNotFoundError(f"No checkpoint directory for seed {seed}")

    files = [
        p for p in seed_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in {".pt", ".pth", ".ckpt"}
    ]
    best = [p for p in files if "best" in p.name.lower()]
    candidates = best if best else files

    if len(candidates) != 1:
        raise RuntimeError(
            f"Expected exactly one best checkpoint for seed {seed}; "
            f"found {len(candidates)} in {seed_dir}: {candidates}"
        )
    return candidates[0]

def extract_state_dict(checkpoint):
    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        return checkpoint["model_state_dict"]
    if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
        return checkpoint["state_dict"]
    if isinstance(checkpoint, dict) and checkpoint and all(torch.is_tensor(v) for v in checkpoint.values()):
        return checkpoint
    raise RuntimeError("Unsupported checkpoint format.")

BASELINE_CHECKPOINTS = {seed: find_best_checkpoint(seed) for seed in SEEDS}

with open(BASELINE_RESULTS_PATH) as f:
    baseline_runs = json.load(f)
baseline_by_seed = {int(r["seed"]): r for r in baseline_runs}

assert set(baseline_by_seed) == set(SEEDS), "Fed-CGM+Time must have all 10 formal seeds."

for seed in SEEDS:
    print(f"seed={seed:>2}: {BASELINE_CHECKPOINTS[seed]}")


seed= 7: /content/drive/MyDrive/dissertation/experiments/7_federated_learning/checkpoints/fed_cgm_time/seed7/best.pt
seed=14: /content/drive/MyDrive/dissertation/experiments/7_federated_learning/checkpoints/fed_cgm_time/seed14/best.pt
seed=21: /content/drive/MyDrive/dissertation/experiments/7_federated_learning/checkpoints/fed_cgm_time/seed21/best.pt
seed=28: /content/drive/MyDrive/dissertation/experiments/7_federated_learning/checkpoints/fed_cgm_time/seed28/best.pt
seed=35: /content/drive/MyDrive/dissertation/experiments/7_federated_learning/checkpoints/fed_cgm_time/seed35/best.pt
seed=42: /content/drive/MyDrive/dissertation/experiments/7_federated_learning/checkpoints/fed_cgm_time/seed42/best.pt
seed=49: /content/drive/MyDrive/dissertation/experiments/7_federated_learning/checkpoints/fed_cgm_time/seed49/best.pt
seed=56: /content/drive/MyDrive/dissertation/experiments/7_federated_learning/checkpoints/fed_cgm_time/seed56/best.pt
seed=63: /content/drive/MyDrive/dissertation/experiments/

## 2. Build the frozen residual model

In [15]:

def load_fed_cgm_time_baseline(seed, device="cpu"):
    base = make_cgm_time_model()
    checkpoint = torch.load(BASELINE_CHECKPOINTS[seed], map_location=device, weights_only=False)
    state = extract_state_dict(checkpoint)

    if state and all(k.startswith("module.") for k in state):
        state = {k[len("module."):]: v for k, v in state.items()}

    base.load_state_dict(state, strict=True)
    base.to(device)
    base.eval()

    for p in base.parameters():
        p.requires_grad = False

    return base

def make_residual_model(seed, encoder, device="cpu"):
    spec = ENCODERS[encoder]
    base = load_fed_cgm_time_baseline(seed, device=device)

    model = FrozenImageOnlyResidualPredictor(
        base_model=base,
        img_feat_dim=spec["img_feat_dim"],
        img_proj_dim=spec["img_proj_dim"],
        img_dropout=spec["img_dropout"],
    )

    # Enforce the intended design even if the class implementation changes:
    # the entire Fed-CGM+Time base must remain frozen.
    if hasattr(model, "base_model"):
        for p in model.base_model.parameters():
            p.requires_grad = False

    return model

# Strict architecture / freezing smoke test for all four encoders.
for encoder in ENCODERS:
    m = make_residual_model(SEEDS[0], encoder)
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    frozen = sum(p.numel() for p in m.parameters() if not p.requires_grad)
    assert trainable > 0
    assert frozen > 0
    print(f"{encoder:>8}: trainable={trainable:,}, frozen={frozen:,}")
    del m


resnet18: trainable=2,217, frozen=1,026,261
     vit: trainable=3,241, frozen=1,026,261
    clip: trainable=3,241, frozen=1,026,261
  dinov2: trainable=3,241, frozen=1,026,261


## 3. Shared helpers, participant-level evaluation, and alignment checks

In [16]:

stats = data.load_cgm_stats()
test_participant_ids = data.load_multimodal_participant_ids("test")
TEST_PARTICIPANTS = sorted(set(test_participant_ids))

aligned_train_samples = data.load_multimodal_samples("train")
assert not any(s["has_meal"] and not s["has_image"] for s in aligned_train_samples)

def forward_fn(model, batch, device):
    cgm, tsm, has_meal, img_feat, target = batch
    pred = model(
        cgm.to(device),
        tsm.to(device),
        img_feat.to(device),
        has_meal.to(device),
    )
    return pred, target.to(device)

def to_jsonable_results(d):
    return {str(k): v for k, v in d.items()}

def to_jsonable_per_participant(d):
    return {
        str(pid): {str(h): v for h, v in per_h.items()}
        for pid, per_h in d.items()
    }

def evaluate_residual(model, loader, participant_ids=None):
    y_pred, y_true = train.predict(
        model, loader, device=DEVICE, forward_fn=forward_fn
    )
    overall = metrics.evaluate(
        y_pred, y_true, stats["cgm_mean"], stats["cgm_std"]
    )
    pp = None
    if participant_ids is not None:
        pp = metrics.evaluate_per_participant(
            y_pred, y_true, participant_ids,
            stats["cgm_mean"], stats["cgm_std"]
        )
    return overall, pp

def load_encoder_data(encoder):
    spec = ENCODERS[encoder]

    clients = fed.load_federated_clients(
        "train", encoder_name=encoder, batch_size=BATCH_SIZE
    )
    val_loader = data.load_multimodal_tensors(
        "val", encoder_name=encoder, batch_size=BATCH_SIZE, shuffle=False
    )
    test_loader = data.load_multimodal_tensors(
        "test", encoder_name=encoder, batch_size=BATCH_SIZE, shuffle=False
    )

    assert len(clients) == 30
    assert sum(len(loader.dataset) for loader in clients.values()) == len(aligned_train_samples)
    assert len(test_loader.dataset) == len(test_participant_ids)

    # Check actual feature dimension.
    batch = next(iter(val_loader))
    img_feat = batch[3]
    assert img_feat.shape[-1] == spec["img_feat_dim"], (
        f'{encoder}: expected {spec["img_feat_dim"]}, got {img_feat.shape[-1]}'
    )

    return clients, val_loader, test_loader

def atomic_json_dump(obj, path):
    path = Path(path)
    tmp = path.with_suffix(".tmp.json")
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2)
    tmp.replace(path)

print("Aligned train samples:", len(aligned_train_samples))
print("Test participants:", TEST_PARTICIPANTS)
print("Participant-level evaluation: ENABLED")


Aligned train samples: 82681
Test participants: [np.str_('CGMacros-008'), np.str_('CGMacros-014'), np.str_('CGMacros-019'), np.str_('CGMacros-020'), np.str_('CGMacros-034'), np.str_('CGMacros-039')]
Participant-level evaluation: ENABLED


## 4. Shared formal training function

In [17]:

def run_encoder(encoder):
    spec = ENCODERS[encoder]
    display_name = spec["display_name"]

    result_dir = RESULTS_ROOT / encoder
    checkpoint_dir = CHECKPOINT_ROOT / encoder
    result_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    run_results_path = result_dir / "run_results.json"

    client_loaders, val_loader, test_loader = load_encoder_data(encoder)

    existing = []
    if run_results_path.exists():
        with open(run_results_path) as f:
            existing = json.load(f)

    completed = {int(r["seed"]) for r in existing}
    print(f"\n{display_name}: already completed seeds = {sorted(completed)}")

    for seed in SEEDS:
        if seed in completed:
            print(f"SKIP {display_name} seed={seed}")
            continue

        print("\n" + "=" * 90)
        print(f"Fed-CGM+Time+{display_name} | seed={seed}")
        print("=" * 90)

        print(
            f"Optimizer: lr={spec['lr']}, "
            f"weight_decay={spec['weight_decay']}"
        )

        fed.set_seed(seed)

        # Critical paired design:
        # this seed uses the SAME seed's frozen Fed-CGM+Time checkpoint.
        def model_fn(seed=seed, encoder=encoder):
            return make_residual_model(seed, encoder, device="cpu")

        seed_checkpoint_dir = checkpoint_dir / f"seed{seed}"

        model, history = fed.federated_train(
            model_fn=model_fn,
            client_loaders=client_loaders,
            val_loader=val_loader,
            cgm_mean=stats["cgm_mean"],
            cgm_std=stats["cgm_std"],
            device=DEVICE,

            lr=spec["lr"],
            weight_decay=spec["weight_decay"],

            max_grad_norm=MAX_GRAD_NORM,
            local_epochs=LOCAL_EPOCHS,
            max_rounds=MAX_ROUNDS,
            clients_per_round=CLIENTS_PER_ROUND,
            early_stop_patience=EARLY_STOP_PATIENCE,
            lr_patience=LR_PATIENCE,
            lr_factor=LR_FACTOR,
            checkpoint_dir=seed_checkpoint_dir,
            verbose=False,
            progress_bar=True,
            progress_desc=f"{encoder} seed={seed}",
            forward_fn=forward_fn,
        )

        val_results, _ = evaluate_residual(model, val_loader)
        test_results, test_pp = evaluate_residual(
            model, test_loader, test_participant_ids
        )

        best_round = int(np.argmin(history["val_loss"]) + 1)

        payload = {
            "model": f"Fed-CGM+Time+{display_name}",
            "encoder": encoder,
            "seed": int(seed),
            "base_model": "Fed-CGM+Time",
            "base_seed": int(seed),
            "base_checkpoint": str(BASELINE_CHECKPOINTS[seed]),
            "base_frozen": True,
            "image_architecture": {
                "img_feat_dim": spec["img_feat_dim"],
                "img_proj_dim": spec["img_proj_dim"],
                "img_dropout": spec["img_dropout"],
            },
            "fl_config": {
                "local_epochs": LOCAL_EPOCHS,
                "clients_per_round": CLIENTS_PER_ROUND,
                "early_stop_patience": EARLY_STOP_PATIENCE,
                "max_rounds": MAX_ROUNDS,
                "lr_patience": LR_PATIENCE,
                "lr_factor": LR_FACTOR,

                "lr": spec["lr"],
                "weight_decay": spec["weight_decay"],

                "batch_size": BATCH_SIZE,
                "max_grad_norm": MAX_GRAD_NORM,
            },
            "n_rounds_trained": len(history["val_loss"]),
            "best_round": best_round,
            "best_val_loss": float(np.min(history["val_loss"])),
            "val_results": to_jsonable_results(val_results),
            "test_results": to_jsonable_results(test_results),
            "test_results_per_participant": to_jsonable_per_participant(test_pp),
        }

        existing.append(payload)
        completed.add(seed)
        atomic_json_dump(existing, run_results_path)

        print(f"SAVED {display_name} seed={seed}")
        print("best round:", best_round)
        print(
            "test RMSE:",
            [round(float(payload["test_results"][str(h)]["rmse"]), 3)
             for h in HORIZONS]
        )

        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print(f"\n{display_name}: completed seeds = {sorted(completed)}")
    return run_results_path


## 5. ResNet18 — 10 formal federated seeds

In [18]:
RESNET18_RESULTS = run_encoder("resnet18")


ResNet18: already completed seeds = []

Fed-CGM+Time+ResNet18 | seed=7
Optimizer: lr=0.001, weight_decay=1e-05
device: cuda  model: FrozenImageOnlyResidualPredictor  params: 1,028,478  trainable: 2,217  clients: 30 (15/30 sampled per round)


resnet18 seed=7:   0%|          | 0/100 [00:00<?, ?it/s]

SAVED ResNet18 seed=7
best round: 15
test RMSE: [12.32, 20.206, 31.327, 38.666, 43.167]

Fed-CGM+Time+ResNet18 | seed=14
Optimizer: lr=0.001, weight_decay=1e-05
device: cuda  model: FrozenImageOnlyResidualPredictor  params: 1,028,478  trainable: 2,217  clients: 30 (15/30 sampled per round)


resnet18 seed=14:   0%|          | 0/100 [00:00<?, ?it/s]

SAVED ResNet18 seed=14
best round: 5
test RMSE: [12.505, 20.312, 31.024, 38.032, 42.292]

Fed-CGM+Time+ResNet18 | seed=21
Optimizer: lr=0.001, weight_decay=1e-05
device: cuda  model: FrozenImageOnlyResidualPredictor  params: 1,028,478  trainable: 2,217  clients: 30 (15/30 sampled per round)


...

SAVED ResNet18 seed=56
best round: 8
test RMSE: [11.984, 19.846, 30.75, 38.287, 42.699]

Fed-CGM+Time+ResNet18 | seed=63
Optimizer: lr=0.001, weight_decay=1e-05
device: cuda  model: FrozenImageOnlyResidualPredictor  params: 1,028,478  trainable: 2,217  clients: 30 (15/30 sampled per round)


resnet18 seed=63:   0%|          | 0/100 [00:00<?, ?it/s]

SAVED ResNet18 seed=63
best round: 13
test RMSE: [11.966, 19.881, 30.826, 37.771, 42.085]

Fed-CGM+Time+ResNet18 | seed=70
Optimizer: lr=0.001, weight_decay=1e-05
device: cuda  model: FrozenImageOnlyResidualPredictor  params: 1,028,478  trainable: 2,217  clients: 30 (15/30 sampled per round)


resnet18 seed=70:   0%|          | 0/100 [00:00<?, ?it/s]

SAVED ResNet18 seed=70
best round: 15
test RMSE: [12.386, 19.989, 30.669, 37.849, 42.01]

ResNet18: completed seeds = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]


## 6. ViT-B/16 — 10 formal federated seeds

In [19]:
VIT_RESULTS = run_encoder("vit")


ViT-B/16: already completed seeds = []

Fed-CGM+Time+ViT-B/16 | seed=7
Optimizer: lr=0.001, weight_decay=1e-05
device: cuda  model: FrozenImageOnlyResidualPredictor  params: 1,029,502  trainable: 3,241  clients: 30 (15/30 sampled per round)


vit seed=7:   0%|          | 0/100 [00:00<?, ?it/s]

SAVED ViT-B/16 seed=7
best round: 2
test RMSE: [12.315, 20.214, 31.367, 38.708, 43.155]

Fed-CGM+Time+ViT-B/16 | seed=14
Optimizer: lr=0.001, weight_decay=1e-05
device: cuda  model: FrozenImageOnlyResidualPredictor  params: 1,029,502  trainable: 3,241  clients: 30 (15/30 sampled per round)


vit seed=14:   0%|          | 0/100 [00:00<?, ?it/s]

SAVED ViT-B/16 seed=14
best round: 2
test RMSE: [12.479, 20.316, 31.083, 38.103, 42.381]

Fed-CGM+Time+ViT-B/16 | seed=21
Optimizer: lr=0.001, weight_decay=1e-05
device: cuda  model: FrozenImageOnlyResidualPredictor  params: 1,029,502  trainable: 3,241  clients: 30 (15/30 sampled per round)


...

SAVED ViT-B/16 seed=56
best round: 4
test RMSE: [12.001, 19.901, 30.771, 38.211, 42.537]

Fed-CGM+Time+ViT-B/16 | seed=63
Optimizer: lr=0.001, weight_decay=1e-05
device: cuda  model: FrozenImageOnlyResidualPredictor  params: 1,029,502  trainable: 3,241  clients: 30 (15/30 sampled per round)


vit seed=63:   0%|          | 0/100 [00:00<?, ?it/s]

SAVED ViT-B/16 seed=63
best round: 2
test RMSE: [11.991, 19.782, 30.653, 37.659, 42.086]

Fed-CGM+Time+ViT-B/16 | seed=70
Optimizer: lr=0.001, weight_decay=1e-05
device: cuda  model: FrozenImageOnlyResidualPredictor  params: 1,029,502  trainable: 3,241  clients: 30 (15/30 sampled per round)


vit seed=70:   0%|          | 0/100 [00:00<?, ?it/s]

SAVED ViT-B/16 seed=70
best round: 3
test RMSE: [12.4, 20.045, 30.813, 38.045, 42.199]

ViT-B/16: completed seeds = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]


## 7. CLIP ViT-B/16 — 10 formal federated seeds

In [20]:
CLIP_RESULTS = run_encoder("clip")


CLIP ViT-B/16: already completed seeds = []

Fed-CGM+Time+CLIP ViT-B/16 | seed=7
Optimizer: lr=0.001, weight_decay=1e-05
device: cuda  model: FrozenImageOnlyResidualPredictor  params: 1,029,502  trainable: 3,241  clients: 30 (15/30 sampled per round)


clip seed=7:   0%|          | 0/100 [00:00<?, ?it/s]

SAVED CLIP ViT-B/16 seed=7
best round: 7
test RMSE: [12.314, 20.176, 31.239, 38.541, 42.984]

Fed-CGM+Time+CLIP ViT-B/16 | seed=14
Optimizer: lr=0.001, weight_decay=1e-05
device: cuda  model: FrozenImageOnlyResidualPredictor  params: 1,029,502  trainable: 3,241  clients: 30 (15/30 sampled per round)


clip seed=14:   0%|          | 0/100 [00:00<?, ?it/s]

SAVED CLIP ViT-B/16 seed=14
best round: 8
test RMSE: [12.387, 20.187, 30.87, 37.868, 42.097]

Fed-CGM+Time+CLIP ViT-B/16 | seed=21
Optimizer: lr=0.001, weight_decay=1e-05
device: cuda  model: FrozenImageOnlyResidualPredictor  params: 1,029,502  trainable: 3,241  clients: 30 (15/30 sampled per round)


...

SAVED CLIP ViT-B/16 seed=56
best round: 8
test RMSE: [12.023, 19.897, 30.727, 38.217, 42.609]

Fed-CGM+Time+CLIP ViT-B/16 | seed=63
Optimizer: lr=0.001, weight_decay=1e-05
device: cuda  model: FrozenImageOnlyResidualPredictor  params: 1,029,502  trainable: 3,241  clients: 30 (15/30 sampled per round)


clip seed=63:   0%|          | 0/100 [00:00<?, ?it/s]

SAVED CLIP ViT-B/16 seed=63
best round: 7
test RMSE: [11.944, 19.763, 30.636, 37.659, 42.085]

Fed-CGM+Time+CLIP ViT-B/16 | seed=70
Optimizer: lr=0.001, weight_decay=1e-05
device: cuda  model: FrozenImageOnlyResidualPredictor  params: 1,029,502  trainable: 3,241  clients: 30 (15/30 sampled per round)


clip seed=70:   0%|          | 0/100 [00:00<?, ?it/s]

SAVED CLIP ViT-B/16 seed=70
best round: 10
test RMSE: [12.367, 19.952, 30.625, 37.869, 42.088]

CLIP ViT-B/16: completed seeds = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]


## 8. DINOv2 ViT-B/14 — 10 formal federated seeds

In [21]:
DINOV2_RESULTS = run_encoder("dinov2")


DINOv2 ViT-B/14: already completed seeds = []

Fed-CGM+Time+DINOv2 ViT-B/14 | seed=7
Optimizer: lr=0.001, weight_decay=1e-05
device: cuda  model: FrozenImageOnlyResidualPredictor  params: 1,029,502  trainable: 3,241  clients: 30 (15/30 sampled per round)


dinov2 seed=7:   0%|          | 0/100 [00:00<?, ?it/s]

SAVED DINOv2 ViT-B/14 seed=7
best round: 9
test RMSE: [12.367, 20.272, 31.401, 38.718, 43.196]

Fed-CGM+Time+DINOv2 ViT-B/14 | seed=14
Optimizer: lr=0.001, weight_decay=1e-05
device: cuda  model: FrozenImageOnlyResidualPredictor  params: 1,029,502  trainable: 3,241  clients: 30 (15/30 sampled per round)


dinov2 seed=14:   0%|          | 0/100 [00:00<?, ?it/s]

SAVED DINOv2 ViT-B/14 seed=14
best round: 6
test RMSE: [12.451, 20.196, 30.845, 37.897, 42.203]

Fed-CGM+Time+DINOv2 ViT-B/14 | seed=21
Optimizer: lr=0.001, weight_decay=1e-05
device: cuda  model: FrozenImageOnlyResidualPredictor  params: 1,029,502  trainable: 3,241  clients: 30 (15/30 sampled per round)


...

SAVED DINOv2 ViT-B/14 seed=56
best round: 20
test RMSE: [11.978, 19.801, 30.589, 38.128, 42.558]

Fed-CGM+Time+DINOv2 ViT-B/14 | seed=63
Optimizer: lr=0.001, weight_decay=1e-05
device: cuda  model: FrozenImageOnlyResidualPredictor  params: 1,029,502  trainable: 3,241  clients: 30 (15/30 sampled per round)


dinov2 seed=63:   0%|          | 0/100 [00:00<?, ?it/s]

SAVED DINOv2 ViT-B/14 seed=63
best round: 7
test RMSE: [11.958, 19.741, 30.539, 37.507, 41.927]

Fed-CGM+Time+DINOv2 ViT-B/14 | seed=70
Optimizer: lr=0.001, weight_decay=1e-05
device: cuda  model: FrozenImageOnlyResidualPredictor  params: 1,029,502  trainable: 3,241  clients: 30 (15/30 sampled per round)


dinov2 seed=70:   0%|          | 0/100 [00:00<?, ?it/s]

SAVED DINOv2 ViT-B/14 seed=70
best round: 8
test RMSE: [12.427, 20.031, 30.659, 37.806, 41.971]

DINOv2 ViT-B/14: completed seeds = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]


## 9. Aggregate four encoders and paired comparison with Fed-CGM+Time

In [25]:

def get_h(d, h):
    if str(h) in d:
        return d[str(h)]
    return d[h]

all_encoder_runs = {}
for encoder in ENCODERS:
    p = RESULTS_ROOT / encoder / "run_results.json"
    assert p.exists(), f"Missing results for {encoder}: {p}"
    with open(p) as f:
        runs = json.load(f)
    assert {int(r["seed"]) for r in runs} == set(SEEDS), f"{encoder}: incomplete seeds"
    all_encoder_runs[encoder] = {int(r["seed"]): r for r in runs}

summary_rows = []
paired_rows = []

for encoder, spec in ENCODERS.items():
    for h in HORIZONS:
        mm_rmse = np.asarray([
            float(get_h(all_encoder_runs[encoder][seed]["test_results"], h)["rmse"])
            for seed in SEEDS
        ])
        mm_mae = np.asarray([
            float(get_h(all_encoder_runs[encoder][seed]["test_results"], h)["mae"])
            for seed in SEEDS
        ])
        base_rmse = np.asarray([
            float(get_h(baseline_by_seed[seed]["test_results"], h)["rmse"])
            for seed in SEEDS
        ])
        base_mae = np.asarray([
            float(get_h(baseline_by_seed[seed]["test_results"], h)["mae"])
            for seed in SEEDS
        ])

        delta_rmse = mm_rmse - base_rmse
        delta_mae = mm_mae - base_mae

        summary_rows.append({
            "encoder": encoder,
            "display_name": spec["display_name"],
            "horizon_min": h,
            "rmse_mean": mm_rmse.mean(),
            "rmse_sd": mm_rmse.std(ddof=1),
            "mae_mean": mm_mae.mean(),
            "mae_sd": mm_mae.std(ddof=1),
            "delta_rmse_vs_fed_cgm_time_mean": delta_rmse.mean(),
            "delta_rmse_vs_fed_cgm_time_sd": delta_rmse.std(ddof=1),
            "delta_mae_vs_fed_cgm_time_mean": delta_mae.mean(),
            "delta_mae_vs_fed_cgm_time_sd": delta_mae.std(ddof=1),
        })

        for i, seed in enumerate(SEEDS):
            paired_rows.append({
                "encoder": encoder,
                "seed": seed,
                "horizon_min": h,
                "fed_cgm_time_rmse": base_rmse[i],
                "fed_multimodal_rmse": mm_rmse[i],
                "delta_rmse": delta_rmse[i],
                "fed_cgm_time_mae": base_mae[i],
                "fed_multimodal_mae": mm_mae[i],
                "delta_mae": delta_mae[i],
            })

summary_df = pd.DataFrame(summary_rows)
paired_df = pd.DataFrame(paired_rows)

summary_path = SUMMARY_ROOT / "fed_multimodal_four_encoder_summary.csv"
paired_path = SUMMARY_ROOT / "fed_multimodal_paired_seed_comparison.csv"

summary_df.to_csv(summary_path, index=False)
paired_df.to_csv(paired_path, index=False)

display(summary_df)

print("\nInterpretation of delta:")
print("  delta < 0  => image model has lower error than paired Fed-CGM+Time")
print("  delta > 0  => image model has higher error than paired Fed-CGM+Time")
print("\nSaved:")
print(summary_path)
print(paired_path)


,encoder,display_name,horizon_min,rmse_mean,rmse_sd,mae_mean,mae_sd,delta_rmse_vs_fed_cgm_time_mean,delta_rmse_vs_fed_cgm_time_sd,delta_mae_vs_fed_cgm_time_mean,delta_mae_vs_fed_cgm_time_sd
0,resnet18,ResNet18,15,12.343239,0.284660,8.500001,0.273939,-0.039206,0.046584,0.005475,0.022900
1,resnet18,ResNet18,30,20.156132,0.271997,13.652096,0.290711,-0.062569,0.085641,0.014981,0.057382
2,resnet18,ResNet18,60,31.022200,0.227708,20.485869,0.248198,-0.056556,0.141290,0.042184,0.137169
3,resnet18,ResNet18,90,38.072606,0.280492,24.998250,0.337882,-0.014538,0.157886,0.027130,0.133523
4,resnet18,ResNet18,120,42.374801,0.341505,27.908866,0.381320,-0.002210,0.154320,-0.011345,0.119808
5,vit,ViT-B/16,15,12.359787,0.291991,8.499259,0.258338,-0.022658,0.030251,0.004732,0.024963
6,vit,ViT-B/16,30,20.167805,0.297110,13.650108,0.258020,-0.050896,0.046135,0.012993,0.050039
7,vit,ViT-B/16,60,31.018720,0.259197,20.473425,0.194154,-0.060036,0.068128,0.029740,0.123004
8,vit,ViT-B/16,90,38.053116,0.306113,24.988999,0.283141,-0.034029,0.086468,0.017878,0.147892
9,vit,ViT-B/16,120,42.348305,0.314941,27.912558,0.355436,-0.028706,0.097085,-0.007654,0.148011



Interpretation of delta:
  delta < 0  => image model has lower error than paired Fed-CGM+Time
  delta > 0  => image model has higher error than paired Fed-CGM+Time

Saved:
/content/drive/MyDrive/dissertation/experiments/7_federated_learning/results/fed_cgm_time_image/summary/fed_multimodal_four_encoder_summary.csv
/content/drive/MyDrive/dissertation/experiments/7_federated_learning/results/fed_cgm_time_image/summary/fed_multimodal_paired_seed_comparison.csv


## 10. Participant-level paired image effect

In [26]:

participant_rows = []

for encoder, spec in ENCODERS.items():
    for seed in SEEDS:
        mm_pp = all_encoder_runs[encoder][seed]["test_results_per_participant"]
        base_pp = baseline_by_seed[seed]["test_results_per_participant"]

        for pid in TEST_PARTICIPANTS:
            pid_key = str(pid)
            assert pid_key in mm_pp, f"{encoder}, seed {seed}: missing participant {pid}"
            assert pid_key in base_pp, f"baseline seed {seed}: missing participant {pid}"

            for h in HORIZONS:
                mm_h = get_h(mm_pp[pid_key], h)
                base_h = get_h(base_pp[pid_key], h)

                participant_rows.append({
                    "encoder": encoder,
                    "display_name": spec["display_name"],
                    "seed": seed,
                    "participant_id": pid,
                    "horizon_min": h,
                    "fed_cgm_time_rmse": float(base_h["rmse"]),
                    "fed_multimodal_rmse": float(mm_h["rmse"]),
                    "delta_rmse": float(mm_h["rmse"]) - float(base_h["rmse"]),
                    "fed_cgm_time_mae": float(base_h["mae"]),
                    "fed_multimodal_mae": float(mm_h["mae"]),
                    "delta_mae": float(mm_h["mae"]) - float(base_h["mae"]),
                })

participant_df = pd.DataFrame(participant_rows)
participant_path = SUMMARY_ROOT / "fed_multimodal_participant_level_paired.csv"
participant_df.to_csv(participant_path, index=False)

participant_summary = (
    participant_df
    .groupby(["encoder", "display_name", "participant_id", "horizon_min"], as_index=False)
    .agg(
        delta_rmse_mean=("delta_rmse", "mean"),
        delta_rmse_sd=("delta_rmse", "std"),
        delta_mae_mean=("delta_mae", "mean"),
        delta_mae_sd=("delta_mae", "std"),
    )
)
participant_summary_path = (
    SUMMARY_ROOT / "fed_multimodal_participant_level_summary.csv"
)
participant_summary.to_csv(participant_summary_path, index=False)

display(participant_summary)

print("\nSaved participant-level paired results:")
print(participant_path)
print(participant_summary_path)


,encoder,display_name,participant_id,horizon_min,delta_rmse_mean,delta_rmse_sd,delta_mae_mean,delta_mae_sd
0,clip,CLIP ViT-B/16,CGMacros-008,15,0.038089,0.045971,0.048209,0.049922
1,clip,CLIP ViT-B/16,CGMacros-008,30,0.096410,0.121366,0.072494,0.115119
2,clip,CLIP ViT-B/16,CGMacros-008,60,0.233835,0.386046,0.124970,0.276553
3,clip,CLIP ViT-B/16,CGMacros-008,90,0.412438,0.602663,0.140736,0.408910
4,clip,CLIP ViT-B/16,CGMacros-008,120,0.454854,0.797756,0.134212,0.527477
...,...,...,...,...,...,...,...,...
115,vit,ViT-B/16,CGMacros-039,15,-0.140934,0.240688,-0.075279,0.146221
116,vit,ViT-B/16,CGMacros-039,30,-0.308925,0.360982,-0.202738,0.256815
117,vit,ViT-B/16,CGMacros-039,60,-0.325822,0.590102,-0.310045,0.491423
118,vit,ViT-B/16,CGMacros-039,90,-0.163654,0.695136,-0.226307,0.604636



Saved participant-level paired results:
/content/drive/MyDrive/dissertation/experiments/7_federated_learning/results/fed_cgm_time_image/summary/fed_multimodal_participant_level_paired.csv
/content/drive/MyDrive/dissertation/experiments/7_federated_learning/results/fed_cgm_time_image/summary/fed_multimodal_participant_level_summary.csv


## 11. Final audit

In [27]:

print("=" * 90)
print("FINAL FEDERATED MULTIMODAL AUDIT")
print("=" * 90)

print(f"Training clients: 30 participants")
print(f"Clients per round: {CLIENTS_PER_ROUND}/30")
print(f"Local epochs: {LOCAL_EPOCHS}")
print(f"Early-stop patience: {EARLY_STOP_PATIENCE}")
print(f"Seeds: {SEEDS}")
print(f"Test participants: {TEST_PARTICIPANTS}")
print(f"Encoders: {[ENCODERS[e]['display_name'] for e in ENCODERS]}")

for encoder in ENCODERS:
    p = RESULTS_ROOT / encoder / "run_results.json"
    with open(p) as f:
        runs = json.load(f)
    print(f"{encoder:>8}: {len(runs)}/10 seeds saved")

required = [
    SUMMARY_ROOT / "fed_multimodal_four_encoder_summary.csv",
    SUMMARY_ROOT / "fed_multimodal_paired_seed_comparison.csv",
    SUMMARY_ROOT / "fed_multimodal_participant_level_paired.csv",
    SUMMARY_ROOT / "fed_multimodal_participant_level_summary.csv",
]
for p in required:
    assert p.exists(), f"Missing summary: {p}"
    print("FOUND:", p)

print("\nFormal Fed-CGM+Time+Image experiment complete.")


FINAL FEDERATED MULTIMODAL AUDIT
Training clients: 30 participants
Clients per round: 15/30
Local epochs: 5
Early-stop patience: 20
Seeds: [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]
Test participants: [np.str_('CGMacros-008'), np.str_('CGMacros-014'), np.str_('CGMacros-019'), np.str_('CGMacros-020'), np.str_('CGMacros-034'), np.str_('CGMacros-039')]
Encoders: ['ResNet18', 'ViT-B/16', 'CLIP ViT-B/16', 'DINOv2 ViT-B/14']
resnet18: 10/10 seeds saved
     vit: 10/10 seeds saved
    clip: 10/10 seeds saved
  dinov2: 10/10 seeds saved
FOUND: /content/drive/MyDrive/dissertation/experiments/7_federated_learning/results/fed_cgm_time_image/summary/fed_multimodal_four_encoder_summary.csv
FOUND: /content/drive/MyDrive/dissertation/experiments/7_federated_learning/results/fed_cgm_time_image/summary/fed_multimodal_paired_seed_comparison.csv
FOUND: /content/drive/MyDrive/dissertation/experiments/7_federated_learning/results/fed_cgm_time_image/summary/fed_multimodal_participant_level_paired.csv
FOUND: /c